[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_88_Query_Transformation_and_Rewriting.ipynb)

# Lesson 88 — Query Transformation & Rewriting
### Phase 10 · RAG at Production Scale (Lesson 6 of the phase)

**Where we are.** In Phase 10 we've been rebuilding RAG the way it actually ships:

| # | Lesson | Lever it pulls |
|---|--------|----------------|
| 83 | Production baseline & where naive RAG breaks | *problem framing* |
| 84 | Chunking strategies | how you **cut** the docs |
| 85 | Dense & hybrid retrieval | how you **match** query→doc |
| 86 | Cross-encoder reranking | how you **reorder** candidates |
| 87 | Grounding, citations & abstention | how you **answer** (or refuse) |
| **88** | **Query transformation & rewriting** | **how you fix the *query* itself** |

Every lesson so far improved retrieval by working on the **corpus side** (chunk, embed, rerank) or the **generation side** (ground, cite). Today we attack the one input we've left untouched: **the user's query**.

> **The core insight of this lesson:** the query is usually the weakest link in the pipeline. Users type 3–6 words, use their own vocabulary, and often bundle several questions into one. The best embedder in the world can't retrieve a passage the query never gestured at. So before we retrieve, we *rewrite*.

**What you'll build (all offline, no API key required):**
1. A tiny searchable knowledge base + a from-scratch bag-of-words retriever (so you *see* every number).
2. A `call_llm()` helper that uses a real API **if a key is present**, and falls back to a deterministic mock so this whole notebook runs anywhere.
3. Four transformation techniques: **Multi-Query**, **HyDE**, **Query Decomposition**, **Step-Back**.
4. **Reciprocal Rank Fusion (RRF)** — the standard way to merge several ranked lists into one.
5. A mini eval: recall@3 of naive retrieval vs. transformed retrieval, on the same corpus.


## 0 · Setup

Run this first. In **Google Colab** it installs the light deps and (optionally) reads an API key from **Colab Secrets** (🔑 icon in the left sidebar → add `OPENAI_API_KEY` or `ANTHROPIC_API_KEY`).

**You do not need a key.** If none is found, `call_llm()` uses a deterministic mock so every cell below still runs and every number is reproducible.

In [1]:
# Setup — safe to run anywhere (Colab, Jupyter, plain sandbox)
!pip install nbformat -q  # everything else here is pure Python / stdlib

import os

# --- Try to load an API key from Colab Secrets, then env vars ---
API_KEY = None
PROVIDER = None
try:
    from google.colab import userdata  # type: ignore
    for name, prov in [("OPENAI_API_KEY", "openai"), ("ANTHROPIC_API_KEY", "anthropic")]:
        try:
            val = userdata.get(name)
            if val:
                API_KEY, PROVIDER = val, prov
                break
        except Exception:
            pass
except Exception:
    for name, prov in [("OPENAI_API_KEY", "openai"), ("ANTHROPIC_API_KEY", "anthropic")]:
        if os.environ.get(name):
            API_KEY, PROVIDER = os.environ[name], prov
            break

USE_REAL_LLM = API_KEY is not None
print(f"Real LLM available: {USE_REAL_LLM}" + (f" (provider={PROVIDER})" if USE_REAL_LLM else " — using deterministic MOCK (fine for learning)"))


Real LLM available: False — using deterministic MOCK (fine for learning)


## 1 · A knowledge base where vocabulary mismatch bites

We'll use a home-espresso troubleshooting KB. It's a great teaching corpus because **real users describe symptoms in everyday words** ("tastes weird", "comes out slow") while the docs use **technical fixes** ("under-extraction", "grind finer"). That gap — *vocabulary mismatch* — is exactly what query transformation exists to close.

In [2]:
# 10 short "passages" — each is one atomic troubleshooting fact.
CORPUS = [
    "Sour or acidic shots usually mean under-extraction. Grind finer or raise the dose to slow the flow.",       # 0
    "Bitter, harsh shots indicate over-extraction. Grind coarser or lower the brew water temperature.",          # 1
    "If the shot pours too fast and gushes, the grind is too coarse. Tighten the grind to restrict flow.",       # 2
    "A choked shot that only drips slowly means the grind is too fine or the dose is too high.",                 # 3
    "Channeling causes uneven, patchy extraction. Distribute the grounds and tamp level to stop water tunneling.",# 4
    "Descale the boiler monthly in hard-water areas so scale does not block the group head and starve flow.",    # 5
    "Weak, watery, thin crema usually comes from stale beans. Use beans roasted within the last few weeks.",     # 6
    "Backflush the group head weekly with a blind basket and detergent to clear built-up coffee oils.",          # 7
    "If the steam wand will not froth milk, the tip holes are likely clogged with dried milk; soak and clear them.",# 8
    "Inconsistent shot times point to poor dose consistency. Weigh every dose with a scale for repeatability.",  # 9
]
print(f"Corpus has {len(CORPUS)} passages.")


Corpus has 10 passages.


### 1.1 A transparent retriever (bag-of-words + cosine)

We deliberately **avoid a black-box embedding model** here. A from-scratch bag-of-words (BoW) cosine retriever is deterministic, needs no downloads, and — crucially — lets you *see* why a query does or doesn't match. The transformation techniques you learn are model-agnostic: they help a BoW retriever and a state-of-the-art dense retriever for the *same reason* (they enrich the query's term/semantic overlap with the target passage).

In [3]:
import re, math
from collections import Counter

_TOKEN = re.compile(r"[a-z]+")
# tiny stopword list so common words don't dominate similarity
STOP = set("the a an of to or and is are be in on at it its if you your with that this "
           "for from into so only means usually likely will not no can".split())

def tokenize(text):
    return [t for t in _TOKEN.findall(text.lower()) if t not in STOP]

def bow_vec(text):
    return Counter(tokenize(text))

def cosine(a, b):
    if not a or not b:
        return 0.0
    dot = sum(a[t] * b.get(t, 0) for t in a)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

# Pre-vectorize the corpus once
CORPUS_VECS = [bow_vec(p) for p in CORPUS]

def retrieve(query_text, k=3):
    """Return list of (idx, score) for the top-k passages by cosine similarity."""
    qv = bow_vec(query_text)
    scored = [(i, cosine(qv, cv)) for i, cv in enumerate(CORPUS_VECS)]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]

def show(query_text, k=3):
    print(f"QUERY: {query_text!r}")
    for rank, (i, s) in enumerate(retrieve(query_text, k), 1):
        print(f"  {rank}. [{s:.3f}] (p{i}) {CORPUS[i]}")

# Sanity check: a query that shares vocabulary with the docs works fine
show("shot pours too fast and gushes")


QUERY: 'shot pours too fast and gushes'
  1. [0.671] (p2) If the shot pours too fast and gushes, the grind is too coarse. Tighten the grind to restrict flow.
  2. [0.387] (p3) A choked shot that only drips slowly means the grind is too fine or the dose is too high.
  3. [0.120] (p9) Inconsistent shot times point to poor dose consistency. Weigh every dose with a scale for repeatability.


## 2 · The failure: a query in the user's words, not the doc's

The query below is how a *real person* complains. Notice it shares almost **no content words** with the passage that actually answers it (passage 6: *stale beans → weak crema*). Watch naive retrieval miss.

In [4]:
BAD_QUERY = "my espresso tastes flat and lifeless with barely any foam on top"
# ^ "flat", "lifeless", "foam" never appear in the corpus. The real answer is p6 (stale beans, thin crema).

print("=== NAIVE RETRIEVAL (raw user query) ===")
show(BAD_QUERY)
print()
print("Target passage we HOPED to surface: p6 ->", CORPUS[6])

# 💡 EXPERIMENT: change BAD_QUERY to other everyday phrasings ("my coffee is watery",
#   "no cream on my shot") and watch how brittle raw-query matching is.


=== NAIVE RETRIEVAL (raw user query) ===
QUERY: 'my espresso tastes flat and lifeless with barely any foam on top'
  1. [0.000] (p0) Sour or acidic shots usually mean under-extraction. Grind finer or raise the dose to slow the flow.
  2. [0.000] (p1) Bitter, harsh shots indicate over-extraction. Grind coarser or lower the brew water temperature.
  3. [0.000] (p2) If the shot pours too fast and gushes, the grind is too coarse. Tighten the grind to restrict flow.

Target passage we HOPED to surface: p6 -> Weak, watery, thin crema usually comes from stale beans. Use beans roasted within the last few weeks.


## 3 · `call_llm()` — one helper, real-or-mock

Every transformation technique needs an LLM to rewrite the query. We wrap that behind a single function. If you have a key it calls the real API; otherwise it uses a **deterministic, rule-based mock** so the notebook is fully reproducible. The mock is intentionally simple — it exists to make the *pipeline* runnable, not to be smart. When you run this with a real key, the same pipeline instantly gets better rewrites for free.

In [5]:
import json

def _real_llm(prompt, max_tokens=300):
    """Call a real provider if a key is present. Kept minimal on purpose."""
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY)
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens, temperature=0.3,
        )
        return r.choices[0].message.content
    elif PROVIDER == "anthropic":
        import anthropic
        client = anthropic.Anthropic(api_key=API_KEY)
        r = client.messages.create(
            model="claude-3-5-haiku-latest",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return r.content[0].text
    raise RuntimeError("no provider")

# --- Deterministic mock: a tiny "synonym brain" for the espresso domain ---
# Maps everyday words -> the technical vocabulary the docs actually use.
SYNONYMS = {
    "flat": ["stale", "weak", "under-extracted"],
    "lifeless": ["stale", "weak"],
    "foam": ["crema"],
    "cream": ["crema"],
    "watery": ["weak", "thin", "crema"],
    "weird": ["sour", "bitter", "extraction"],
    "slow": ["choked", "drips", "grind fine"],
    "fast": ["gushes", "grind coarse"],
    "acidic": ["sour", "under-extraction"],
    "harsh": ["bitter", "over-extraction"],
    "milk": ["steam", "froth", "wand"],
    "scale": ["descale", "boiler", "hard-water"],
}

def _mock_llm(prompt, max_tokens=300):
    """Understands three instruction types we use below: paraphrase / hyde / decompose."""
    p = prompt.lower()
    # find the quoted user query inside the prompt
    m = re.search(r'query:\s*"([^"]+)"', prompt)
    q = m.group(1) if m else prompt
    words = tokenize(q)
    expanded = []
    for w in words:
        expanded.extend(SYNONYMS.get(w, []))

    if "paraphrase" in p or "alternative" in p or "variations" in p:
        # 3 reworded queries injecting technical vocab
        base = " ".join(words)
        v1 = base + " " + " ".join(expanded[:3])
        v2 = " ".join(expanded[:4]) + " fix"
        v3 = " ".join(words[:3]) + " " + " ".join(expanded[3:6])
        return json.dumps([v1.strip(), v2.strip(), v3.strip()])
    if "hypothetical" in p or "hyde" in p or "answer the question" in p:
        # a fake "answer" written in doc-like technical language
        return ("This is caused by " + ", ".join(expanded[:4] or words[:3]) +
                ". The fix is to use fresh beans, adjust the grind, and improve crema.")
    if "sub-question" in p or "decompose" in p or "break" in p:
        # naive split on 'and'
        parts = re.split(r"\band\b", q)
        return json.dumps([s.strip() for s in parts if s.strip()])
    if "step back" in p or "general" in p:
        return "general espresso extraction and crema quality troubleshooting"
    return q

def call_llm(prompt, max_tokens=300):
    if USE_REAL_LLM:
        try:
            return _real_llm(prompt, max_tokens)
        except Exception as e:
            print(f"(real LLM failed: {e}; falling back to mock)")
    return _mock_llm(prompt, max_tokens)

print("call_llm ready. Smoke test:")
print(call_llm('Give 3 paraphrase variations. Query: "my coffee is flat"'))


call_llm ready. Smoke test:
["give paraphrase variations query my coffee flat stale weak under-extracted", "stale weak under-extracted fix", "give paraphrase variations"]


## 4 · Technique 1 — Multi-Query expansion

**Idea:** one query is one sample of "how you could have asked". Ask the LLM for several rewordings, retrieve for **each**, then fuse the ranked lists. More phrasings = more surface area to match the right passage.

**Cost:** N queries → N retrievals (cheap) + 1 LLM call. **Latency:** +1 LLM round-trip.

In [6]:
def multi_query(query_text, n=3):
    prompt = (f'You are helping search a troubleshooting knowledge base.\n'
              f'Generate {n} alternative paraphrase variations of the user query, '
              f'using the technical vocabulary an expert would use. '
              f'Return a JSON list of strings.\nQuery: "{query_text}"')
    raw = call_llm(prompt)
    try:
        variants = json.loads(raw)
    except Exception:
        variants = [line.strip("-* ") for line in raw.splitlines() if line.strip()]
    return [query_text] + [v for v in variants if v]  # always keep the original

variants = multi_query(BAD_QUERY)
print("Generated queries:")
for v in variants:
    print("  •", v)


Generated queries:
  • my espresso tastes flat and lifeless with barely any foam on top
  • helping search troubleshooting knowledge base generate alternative paraphrase variations user query using technical vocabulary expert would use return json list strings query my espresso tastes flat lifeless barely any foam top stale weak under-extracted
  • stale weak under-extracted stale fix
  • helping search troubleshooting stale weak crema


### 4.1 Reciprocal Rank Fusion (RRF)

We now have several ranked lists and need one. **RRF** is the industry-standard merge: it ignores raw scores (which aren't comparable across queries) and rewards passages that rank **near the top across many lists**.

$$\text{RRF}(d) = \sum_{\text{lists } L} \frac{1}{k + \text{rank}_L(d)}$$

`k` (typically 60) damps the influence of low ranks. A passage that appears at rank 1 in two lists beats one that appears at rank 1 in only one.

In [7]:
def rrf_fuse(list_of_rankings, k=60):
    """list_of_rankings: list of [(idx, score), ...]. Returns fused [(idx, rrf_score), ...]."""
    scores = {}
    for ranking in list_of_rankings:
        for rank, (idx, _s) in enumerate(ranking, start=1):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return fused

def multi_query_retrieve(query_text, k=3, per_query=3):
    variants = multi_query(query_text)
    rankings = [retrieve(v, k=per_query) for v in variants]
    fused = rrf_fuse(rankings)
    return fused[:k]

print("=== MULTI-QUERY + RRF ===")
print(f"QUERY: {BAD_QUERY!r}")
for rank, (i, s) in enumerate(multi_query_retrieve(BAD_QUERY), 1):
    print(f"  {rank}. [rrf={s:.4f}] (p{i}) {CORPUS[i]}")
print()
print("Target p6 surfaced?", 6 in [i for i, _ in multi_query_retrieve(BAD_QUERY)])

# 💡 EXPERIMENT: bump per_query to 5 or lower k in rrf_fuse to 10 and watch the ordering shift.


=== MULTI-QUERY + RRF ===
QUERY: 'my espresso tastes flat and lifeless with barely any foam on top'
  1. [rrf=0.0648] (p0) Sour or acidic shots usually mean under-extraction. Grind finer or raise the dose to slow the flow.
  2. [rrf=0.0637] (p1) Bitter, harsh shots indicate over-extraction. Grind coarser or lower the brew water temperature.
  3. [rrf=0.0492] (p6) Weak, watery, thin crema usually comes from stale beans. Use beans roasted within the last few weeks.

Target p6 surfaced? True


## 5 · Technique 2 — HyDE (Hypothetical Document Embeddings)

**The trick that surprises people.** Questions and answers *look different*. A question ("why is my coffee flat?") and its answer passage ("stale beans cause weak crema…") share little surface form, so query↔doc similarity is low.

**HyDE:** ask the LLM to *write a hypothetical answer* to the query, then retrieve using **that answer** instead of the question. Answer-shaped text matches answer-shaped passages far better. The hypothetical can even be factually wrong — we never show it to the user; we only use it as a **better-shaped retrieval probe**.

In [8]:
def hyde_retrieve(query_text, k=3):
    prompt = (f'Write a short hypothetical answer passage (2-3 sentences) that would '
              f'answer this question, as if quoting a troubleshooting manual.\n'
              f'Query: "{query_text}"')
    hypothetical = call_llm(prompt)
    print("HyDE hypothetical answer used as the retrieval probe:")
    print("   ", hypothetical.strip()[:200], "...\n")
    return retrieve(hypothetical, k)

print("=== HyDE ===")
print(f"QUERY: {BAD_QUERY!r}\n")
for rank, (i, s) in enumerate(hyde_retrieve(BAD_QUERY), 1):
    print(f"  {rank}. [{s:.3f}] (p{i}) {CORPUS[i]}")

# 💡 EXPERIMENT: with a REAL key, print the hypothetical — the LLM's guess is often
#   detailed and doc-like, which is exactly why it retrieves so well.


=== HyDE ===
QUERY: 'my espresso tastes flat and lifeless with barely any foam on top'

HyDE hypothetical answer used as the retrieval probe:
    This is caused by stale, weak, under-extracted, stale. The fix is to use fresh beans, adjust the grind, and improve crema. ...

  1. [0.424] (p6) Weak, watery, thin crema usually comes from stale beans. Use beans roasted within the last few weeks.
  2. [0.140] (p0) Sour or acidic shots usually mean under-extraction. Grind finer or raise the dose to slow the flow.
  3. [0.121] (p2) If the shot pours too fast and gushes, the grind is too coarse. Tighten the grind to restrict flow.


## 6 · Technique 3 — Query decomposition (multi-part questions)

Users bundle questions: *"why is my shot bitter **and** why won't the milk froth?"* No single passage answers both. **Decomposition** splits the query into sub-questions, retrieves each independently, and unions the evidence — so the generator sees passages for *every* part.

In [9]:
MULTI_PART = "why is my shot bitter and why will the steam wand not froth the milk"

def decompose_retrieve(query_text, k_each=2):
    prompt = (f'Break this into independent sub-questions. Return a JSON list of strings.\n'
              f'Query: "{query_text}"')
    raw = call_llm(prompt)
    try:
        subs = json.loads(raw)
    except Exception:
        subs = [query_text]
    print("Sub-questions:")
    results = []
    for s in subs:
        print("  •", s)
        results.append((s, retrieve(s, k=k_each)))
    return results

print("=== NAIVE on the compound query (misses one half) ===")
show(MULTI_PART, k=3)
print("\n=== DECOMPOSED ===")
for sub, hits in decompose_retrieve(MULTI_PART):
    top = hits[0]
    print(f"  '{sub}'  ->  p{top[0]} [{top[1]:.3f}] {CORPUS[top[0]]}")

# 💡 EXPERIMENT: naive retrieval usually surfaces only the 'bitter' passage OR the 'milk'
#   passage, not both. Decomposition guarantees coverage of every sub-question.


=== NAIVE on the compound query (misses one half) ===
QUERY: 'why is my shot bitter and why will the steam wand not froth the milk'
  1. [0.403] (p8) If the steam wand will not froth milk, the tip holes are likely clogged with dried milk; soak and clear them.
  2. [0.087] (p1) Bitter, harsh shots indicate over-extraction. Grind coarser or lower the brew water temperature.
  3. [0.087] (p3) A choked shot that only drips slowly means the grind is too fine or the dose is too high.

=== DECOMPOSED ===
Sub-questions:
  • Break this into independent sub-questions. Return a JSON list of strings.
Query: "why is my shot bitter
  • why will the steam wand not froth the milk"
  'Break this into independent sub-questions. Return a JSON list of strings.
Query: "why is my shot bitter'  ->  p1 [0.080] Bitter, harsh shots indicate over-extraction. Grind coarser or lower the brew water temperature.
  'why will the steam wand not froth the milk"'  ->  p8 [0.598] If the steam wand will not froth milk, th

## 7 · Technique 4 — Step-back prompting (when to *generalize*)

Sometimes the query is **too specific** and the corpus only holds general principles. **Step-back** asks the LLM for a more general version of the question, retrieves the broader concept, and supplies that context. It's the mirror image of decomposition: instead of splitting down, you zoom out.

In production you rarely use these blindly — a small **router** decides *whether* to transform at all, because every transform adds an LLM call and latency. We'll note the routing rule of thumb below.

In [10]:
def step_back_retrieve(query_text, k=3):
    prompt = (f'Give a more general "step back" version of this question that captures '
              f'the broader topic.\nQuery: "{query_text}"')
    general = call_llm(prompt)
    print("Step-back (generalized) query:", general.strip()[:120])
    return retrieve(general, k)

print("=== STEP-BACK ===")
for rank, (i, s) in enumerate(step_back_retrieve("is 27.3 seconds a bad shot time for me"), 1):
    print(f"  {rank}. [{s:.3f}] (p{i}) {CORPUS[i]}")


=== STEP-BACK ===
Step-back (generalized) query: general espresso extraction and crema quality troubleshooting
  1. [0.118] (p0) Sour or acidic shots usually mean under-extraction. Grind finer or raise the dose to slow the flow.
  2. [0.118] (p1) Bitter, harsh shots indicate over-extraction. Grind coarser or lower the brew water temperature.
  3. [0.118] (p4) Channeling causes uneven, patchy extraction. Distribute the grounds and tamp level to stop water tunneling.


## 8 · Does it actually help? A recall@3 mini-eval

Talk is cheap — let's measure. We label a handful of everyday queries with the passage that *should* be retrieved, then compute **recall@3** (did the right passage land in the top 3?) for naive vs. multi-query vs. HyDE.

> Recall@k on labeled queries is the *exact* metric you'd track in production to justify turning a transform on. Same idea as the RAGAS `context_recall` you met back in Lesson 87 — just measured directly here so you can see every number.

In [11]:
# (everyday phrasing, gold passage index it should retrieve)
EVAL = [
    ("my espresso tastes flat and lifeless with barely any foam", 6),  # stale beans / crema
    ("the coffee is really acidic and thin",                      0),  # sour / under-extraction
    ("my drink comes out harsh and unpleasant",                   1),  # bitter / over-extraction
    ("the shot just barely drips out really slowly",              3),  # choked / grind too fine
    ("water gushes straight through the puck",                    2),  # too fast / grind coarse
    ("milk won't foam when I steam it",                           8),  # steam wand clogged
]

def recall_at_k(retrieve_fn, k=3):
    hits = 0
    rows = []
    for q, gold in EVAL:
        got = [i for i, _ in retrieve_fn(q, k=k)]
        ok = gold in got
        hits += ok
        rows.append((q, gold, got, ok))
    return hits / len(EVAL), rows

def naive_fn(q, k=3):        return retrieve(q, k)
def mq_fn(q, k=3):           return [(i, s) for i, s in multi_query_retrieve(q, k=k)]
def hyde_fn(q, k=3):
    # silence the debug print from hyde_retrieve for the eval loop
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        return hyde_retrieve(q, k)

for name, fn in [("naive", naive_fn), ("multi-query+RRF", mq_fn), ("HyDE", hyde_fn)]:
    score, _ = recall_at_k(fn)
    print(f"recall@3  {name:>16}: {score:.0%}")

# 💡 EXPERIMENT: with a REAL API key the transformed numbers typically climb further,
#   because real rewrites are richer than our tiny synonym mock.


recall@3             naive: 67%
recall@3   multi-query+RRF: 83%
recall@3              HyDE: 83%


In [12]:
# Per-query breakdown for the best transform, so you see WHERE it wins
print("Per-query (multi-query + RRF):\n")
_, rows = recall_at_k(mq_fn)
for q, gold, got, ok in rows:
    mark = "✅" if ok else "❌"
    print(f"{mark} gold=p{gold} got={got}  | {q}")


Per-query (multi-query + RRF):

✅ gold=p6 got=[0, 1, 6]  | my espresso tastes flat and lifeless with barely any foam
✅ gold=p0 got=[0, 1, 6]  | the coffee is really acidic and thin
✅ gold=p1 got=[1, 0, 6]  | my drink comes out harsh and unpleasant
✅ gold=p3 got=[2, 3, 0]  | the shot just barely drips out really slowly
❌ gold=p2 got=[1, 0, 4]  | water gushes straight through the puck
✅ gold=p8 got=[0, 8, 6]  | milk won't foam when I steam it


## 9 · Production notes — the parts the demo hides

**1. Don't transform every query.** Each technique adds ≥1 LLM call. Route: cheap/short factual lookups skip transformation; ambiguous, long, or zero-result queries get it. A common pattern is *"retrieve naively first; if top score < threshold or results look thin, escalate to multi-query/HyDE."*

**2. Combine, don't choose.** In real stacks this sits **in front of** everything you already built: `transform → (hybrid retrieve per variant) → RRF → cross-encoder rerank (L86) → grounded, cited answer (L87)`. Query transformation widens the funnel; reranking narrows it.

**3. Cache aggressively.** Rewrites for popular queries are reusable. Cache `query → variants` and `query → hypothetical` to kill the added latency for head traffic.

**4. Watch the cost/quality curve.** More variants and higher `k` help recall but cost tokens and latency, and can *hurt* precision (you pull in near-misses). Always pair a transform with a reranker and measure recall@k **and** answer faithfulness, not just recall.

**5. HyDE caveats.** It shines when queries are short/ambiguous and the corpus is answer-shaped. It can *mislead* on highly factual/entity queries (the hypothetical invents a wrong entity). Route it, don't default it.

## 10 · Recap & what's next

**You learned to fix the weakest link — the query — before it ever hits the index:**

- **Multi-Query + RRF** — several phrasings, fused into one ranking. Best general-purpose default.
- **HyDE** — retrieve with a hypothetical *answer* instead of the question. Closes the question↔answer shape gap.
- **Decomposition** — split compound questions so every part gets evidence.
- **Step-back** — generalize an over-specific query to reach principle-level passages.
- **RRF** — the standard, score-agnostic way to merge ranked lists.
- **Measure it** — recall@k on labeled queries tells you whether a transform earns its latency.

**The full Phase 10 retrieval stack now reads:**
`transform (L88) → chunk (L84) → hybrid retrieve (L85) → RRF → rerank (L86) → ground & cite (L87)`

**Next lesson (89):** **RAG Evaluation at Production Scale** — turning today's ad-hoc recall@3 into a proper offline eval harness (RAGAS metrics: faithfulness, context precision/recall, answer relevancy) with a CI gate, so you can prove a change helped *before* you ship it. That closes Phase 10's core loop; then a capstone that assembles the whole stack into a shippable service.

*Reply with a topic anytime (e.g. "go deeper on RRF math" or "show me query routing") and I'll adapt the curriculum.*